In [ ]:
import gin

gin.enter_interactive_mode()

from IPython.display import display, Audio
import torch
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import sys
import os

sys.path.append('..')
torch.set_grad_enabled(False)

# Transform

In [ ]:
from after.autoencoder.audio import StreamableSTFT

sample_rate = 44100
nfft = 512
hop_size = 64
device = "cpu"

time_transform = StreamableSTFT(
    nfft=nfft,
    hop_size=hop_size,
    stream=False,
    normalize=False,
    skip_features=-1,
).to(device)

hop_ms = 1000 * hop_size / sample_rate
window_ms = 1000 * nfft / sample_rate
history_ms = 1000 * (nfft - hop_size) / sample_rate

print(f"STFT: nfft/window={nfft} samples ({window_ms:.2f} ms)")
print(f"Hop: {hop_size} samples ({hop_ms:.2f} ms)")
print(f"Streaming history buffer: {nfft - hop_size} samples ({history_ms:.2f} ms)")


In [3]:
audio_path = "../perso/03_Saw Pad.wav"
if not os.path.exists(audio_path):
    audio_path = "perso/03_Saw Pad.wav"

audio, sr = librosa.load(audio_path, sr=sample_rate, mono=True, duration = 10)
x = torch.from_numpy(audio).float()[None, None].to(device)

features = time_transform(x)
reconstructed = time_transform.inverse(features)

original = x[0, 0].detach().cpu().numpy()
recon = reconstructed[0, 0, :len(original)].detach().cpu().numpy()
    
    
print("original")
display(Audio(original, rate=sr))
print("reconstructed")
display(Audio(recon, rate=sr))

if True:
    real, imag = torch.chunk(features, 2, dim=1)
    spec = torch.complex(real[:, 0], imag[:, 0])[0].detach().cpu().numpy()

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    librosa.display.specshow(
        librosa.amplitude_to_db(np.abs(spec), ref=np.max),
        sr=sr,
        hop_length=hop_size,
        y_axis="linear",
        x_axis="time",
        ax=axes[0],
    )
    axes[0].set_title("Magnitude")

    librosa.display.specshow(
        np.angle(spec),
        sr=sr,
        hop_length=hop_size,
        y_axis="linear",
        x_axis="time",
        ax=axes[1],
        cmap="twilight",
    )
    axes[1].set_title("Phase")
    plt.tight_layout()

In [4]:
import time

repeats = 100
audio_seconds = x.shape[-1] / sr

for _ in range(3):
    features = time_transform(x)
    reconstructed = time_transform.inverse(features)

if device == "cuda":
    torch.cuda.synchronize()

start = time.perf_counter()
for _ in range(repeats):
    features = time_transform(x)
    reconstructed = time_transform.inverse(features)

if device == "cuda":
    torch.cuda.synchronize()

elapsed = (time.perf_counter() - start) / repeats
rtf = elapsed / audio_seconds

print(f"audio duration: {audio_seconds:.3f} s")
print(f"stft + inverse time: {elapsed * 1000:.3f} ms")
print(f"rtf: {rtf:.4f}x")

audio duration: 10.000 s
stft + inverse time: 29.119 ms
rtf: 0.0029x


# Network

In [73]:
from after.autoencoder.networks.SimpleNet2D import AutoEncoder2D
from after.autoencoder.networks.bottlenecks import VAEBottleneck


in_size = 2


encoder_fast =  AutoEncoder2D(in_size = 2,
                 bottleneck_size = 4,
                 channels=[2, 4, 8, 16],
                 time_ratios=[1 ,1, 1, 1],
                 freq_ratios=[2, 2, 2, 2],
                 freq_size = 512,
                 kernel_size= 3,
                 bottleneck=VAEBottleneck(),
                 time_transform=time_transform,
                 use_vae = True)
    
    
audio = torch.randn(1, 1, 131072)
z_f, _ = encoder_fast.encode(audio)


time_transform_slow = StreamableSTFT(
    nfft=1024,
    hop_size=512,
    stream=False,
    normalize=False,
    skip_features=-1,
).to(device)


encoder_slow =  AutoEncoder2D(in_size = 2,
                 bottleneck_size = 16,
                 channels=[16, 32, 64, 64, 64, 64, 64, 128],
                 time_ratios=[1 ,2, 2, 2, 1, 1, 1, 1],
                 freq_ratios=[2, 2, 2, 2, 2, 2, 2, 2],
                 freq_size = 1024,
                 kernel_size= 3,
                 bottleneck=VAEBottleneck(),
                 time_transform=time_transform_slow,
                 use_vae = True)
    
    
z_s, _ = encoder_slow.encode(audio)

In [ ]:
import time

encoders = {
    "fast": encoder_fast,
    "slow": encoder_slow,
}

test_audio = torch.randn(1, 1, 131072)
audio_seconds = test_audio.shape[-1] / sr
repeats = 10

for name, encoder in encoders.items():
    encoder = encoder.to(device).eval()
    params = sum(p.numel() for p in encoder.parameters())

    z, _ = encoder.encode(test_audio)
    compression = test_audio.shape[-1] / z.shape[-1]

    for _ in range(3):
        z, _ = encoder.encode(test_audio)

    start = time.perf_counter()
    for _ in range(repeats):
        z, _ = encoder.encode(test_audio)

    elapsed = (time.perf_counter() - start) / repeats
    rtf = elapsed / audio_seconds

    print(name)
    print(f"  latent shape: {tuple(z.shape)}")
    print(f"  compression ratio: {compression}:1")
    print(f"  parameters: {params:,}")
    print(f"  encode time: {elapsed * 1000:.3f} ms")
    print(f"  encode rtf: {rtf:.4f}x")

In [ ]:
class EncoderOnly(torch.nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def forward(self, x):
        z, _ = self.encoder.encode(x)
        return z

from torchinfo import summary

for name, encoder in encoders.items():
    print(name)
    print(summary(EncoderOnly(encoder).to(device), input_size=tuple(test_audio.shape), device=device, depth=10))


# Decoder

In [125]:

decoder_fast =  AutoEncoder2D(in_size = 2,
                 bottleneck_size = 20,
                 channels=[4, 8, 16, 32, 32, 32, 32],
                 time_ratios=[1 ,1, 1, 1, 1, 1, 1],
                 freq_ratios=[2, 2, 2, 2, 2, 2, 1],
                 freq_size = 512,
                 kernel_size= 3,
                 bottleneck=VAEBottleneck(),
                 time_transform=time_transform,
                 use_vae = True)



In [ ]:
decoder_fast = decoder_fast.to(device).eval()

z_f, _ = encoder_fast.encode(test_audio)
z_s, _ = encoder_slow.encode(test_audio)

repeat = int(np.ceil(z_f.shape[-1] / z_s.shape[-1]))
z_s_repeated = z_s.repeat_interleave(repeat, dim=-1)[..., :z_f.shape[-1]]
z = torch.cat([z_f, z_s_repeated], dim=1)

repeats = 20
audio_seconds = test_audio.shape[-1] / sr

for _ in range(3):
    y = decoder_fast.decode(z)

start = time.perf_counter()
for _ in range(repeats):
    y = decoder_fast.decode(z)

elapsed = (time.perf_counter() - start) / repeats
rtf = elapsed / audio_seconds

params = sum(p.numel() for p in decoder_fast.parameters())


print(f"fast code: {tuple(z_f.shape)}")
print(f"  parameters: {params:,}")
print(f"slow code repeated: {tuple(z_s_repeated.shape)}")
print(f"decoder input: {tuple(z.shape)}")
print(f"decoder output: {tuple(y.shape)}")
print(f"decode time: {elapsed * 1000:.3f} ms")
print(f"decode rtf: {rtf:.4f}x")


In [ ]:
start = time.perf_counter()
for _ in range(repeats):
    h = decoder_fast._decode_features(z)
decode_features_time = (time.perf_counter() - start) / repeats

start = time.perf_counter()
for _ in range(repeats):
    y = decoder_fast.time_transform.inverse(h)
istft_time = (time.perf_counter() - start) / repeats

print(f"decode features: {decode_features_time * 1000:.3f} ms")
print(f"istft: {istft_time * 1000:.3f} ms")

In [ ]:
from torchinfo import summary

class DecoderOnly(torch.nn.Module):
    def __init__(self, decoder):
        super().__init__()
        self.decoder = decoder

    def forward(self, z):
        return self.decoder.decode(z)

decoder_summary = DecoderOnly(decoder_fast).to(device).eval()

summary(
    decoder_summary,
    input_data=z,
    device=device,
    # depth=2,
    # verbose=2,
)

In [ ]:
from collections import defaultdict

def tensor_shape(x):
    if torch.is_tensor(x):
        return tuple(x.shape)
    if isinstance(x, (list, tuple)) and len(x) > 0:
        return tensor_shape(x[0])
    return None

times = defaultdict(float)
counts = defaultdict(int)
shapes = {}
starts = {}
handles = []

def is_leaf(module):
    return len(list(module.children())) == 0

def pre_hook(name):
    def hook(module, inputs):
        starts[name] = time.perf_counter()
    return hook

def post_hook(name):
    def hook(module, inputs, output):
        times[name] += time.perf_counter() - starts[name]
        counts[name] += 1
        shapes[name] = (tensor_shape(inputs), tensor_shape(output))
    return hook

for name, module in decoder_fast.named_modules():
    if name and is_leaf(module):
        handles.append(module.register_forward_pre_hook(pre_hook(name)))
        handles.append(module.register_forward_hook(post_hook(name)))

for _ in range(3):
    h = decoder_fast._decode_features(z)

times.clear()
counts.clear()

profile_repeats = 10
for _ in range(profile_repeats):
    h = decoder_fast._decode_features(z)

for handle in handles:
    handle.remove()

rows = []
for name, total in times.items():
    avg_ms = 1000 * total / counts[name]
    input_shape, output_shape = shapes[name]
    rows.append((avg_ms, name, input_shape, output_shape))

for avg_ms, name, input_shape, output_shape in sorted(rows, reverse=True)[:25]:
    print(f"{avg_ms:8.3f} ms  {name:45s} {input_shape} -> {output_shape}")